In [10]:
# === CELL 1: Import thư viện ===
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import folium
from folium.plugins import HeatMap
import warnings
warnings.filterwarnings('ignore')

# Cấu hình hiển thị
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

In [11]:
# === CELL 2: Tải dữ liệu và kiểm tra ban đầu ===
df = pd.read_csv('../data/melbourne_price_data.csv')

print("=== Kích thước dataset ===")
print(f"Số dòng: {df.shape[0]:,}")
print(f"Số cột: {df.shape[1]}")
print("\n=== 5 dòng đầu ===")
display(df.head())

print("\n=== Thông tin cột ===")
df.info()

print("\n=== Thống kê mô tả (Numeric_Price) ===")
display(df.describe())

=== Kích thước dataset ===
Số dòng: 145,134
Số cột: 20

=== 5 dòng đầu ===


,Property_ID,Status,Full_Address,Suburb,Postcode,Property_Type,Method,Date,Beds,Baths,Car_Spaces,LandSize_sqm,Propertycount,Raw_Price,Numeric_Price,Latitude,Longitude,Distance_to_CBD_km,URL,Last_Updated
0,2019150028,For Sale,"910/188 Ballarat Road, FOOTSCRAY VIC 3011",FOOTSCRAY,3011,Apartment / Unit / Flat,Private Treaty,NaN,2,2,1,0.00,358,"$530,000 - $558,000",544000.00,-37.79,144.89,6.83,https://www.domain.com.au/910-188-ballarat-roa...,2026-03-14
1,2010114712,Sold,"695 Mount Blackwood Road, KOROBEIT VIC 3341",KOROBEIT,3341,House,Private Treaty,10 Dec 2012,3,1,2,0.00,4,Price Withheld,NaN,-37.56,144.35,60.64,https://www.domain.com.au/695-mount-blackwood-...,2026-03-16
2,2011918466,Sold,"70 Lohs Lane, MYRNIONG VIC 3341",MYRNIONG,3341,House,Private Treaty,18 May 2015,4,2,4,80900.00,29,"$805,000",805000.00,-37.56,144.36,59.75,https://www.domain.com.au/70-lohs-lane-myrnion...,2026-03-16
3,2013015017,Sold,"89 Lohs Lane, MYRNIONG VIC 3341",MYRNIONG,3341,House,Private Treaty,03 Oct 2016,3,2,2,283300.00,29,"$725,000",725000.00,-37.56,144.36,59.76,https://www.domain.com.au/89-lohs-lane-myrnion...,2026-03-16
4,2013098108,Sold,"688 Mount Blackwood Road, MYRNIONG VIC 3341",MYRNIONG,3341,Vacant land,Private Treaty,04 May 2017,0,0,0,222600.00,29,"$600,000",600000.00,-37.57,144.36,60.20,https://www.domain.com.au/688-mount-blackwood-...,2026-03-16



=== Thông tin cột ===
<class 'pandas.DataFrame'>
RangeIndex: 145134 entries, 0 to 145133
Data columns (total 20 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Property_ID         145134 non-null  int64  
 1   Status              145134 non-null  str    
 2   Full_Address        145134 non-null  str    
 3   Suburb              145134 non-null  str    
 4   Postcode            145134 non-null  int64  
 5   Property_Type       145055 non-null  str    
 6   Method              145134 non-null  str    
 7   Date                112206 non-null  str    
 8   Beds                145134 non-null  int64  
 9   Baths               145134 non-null  int64  
 10  Car_Spaces          145134 non-null  int64  
 11  LandSize_sqm        145055 non-null  float64
 12  Propertycount       145134 non-null  int64  
 13  Raw_Price           145055 non-null  str    
 14  Numeric_Price       130995 non-null  float64
 15  Latitude            14

,Property_ID,Postcode,Beds,Baths,Car_Spaces,LandSize_sqm,Propertycount,Numeric_Price,Latitude,Longitude,Distance_to_CBD_km
count,145134.00,145134.00,145134.00,145134.00,145134.00,145055.00,145134.00,130995.00,145134.00,145134.00,145134.00
mean,2017928339.19,3387.73,3.17,1.78,1.99,6038.52,721.91,978131.44,-37.93,144.97,35.78
std,47157148.99,354.56,1.35,0.91,1.79,522446.09,624.75,1018084.75,0.25,0.30,18.52
min,1432.00,3000.00,0.00,0.00,0.00,0.00,1.00,1.00,-38.50,144.35,0.00
25%,2018861485.50,3095.00,3.00,1.00,1.00,0.00,252.00,620000.00,-38.13,144.71,21.99
50%,2020074559.00,3217.00,3.00,2.00,2.00,429.00,517.00,765000.00,-37.88,145.01,33.37
75%,2020423536.25,3796.00,4.00,2.00,2.00,721.00,1070.00,1065000.00,-37.74,145.19,49.00
max,2020685100.00,3991.00,52.00,58.00,300.00,197433984.00,2774.00,136600000.00,-37.50,145.50,88.00


In [12]:
# === CELL 3: Missing values & giá trị thiếu ===
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)

print("Top 10 cột có missing values:")
display(pd.DataFrame({'Missing': missing, '%': missing_pct}).head(10))

# Numeric_Price là target → kiểm tra đặc biệt
print(f"\nNumeric_Price missing: {df['Numeric_Price'].isnull().sum()} dòng ({df['Numeric_Price'].isnull().mean()*100:.2f}%)")
print("Các Status có Price Withheld hoặc range:")
display(df[df['Numeric_Price'].isnull()]['Status'].value_counts())

Top 10 cột có missing values:


,Missing,%
Date,32928,22.69
Numeric_Price,14139,9.74
Property_Type,79,0.05
LandSize_sqm,79,0.05
Raw_Price,79,0.05
Full_Address,0,0.00
Status,0,0.00
Property_ID,0,0.00
Suburb,0,0.00
Beds,0,0.00



Numeric_Price missing: 14139 dòng (9.74%)
Các Status có Price Withheld hoặc range:


Status
Sold        9221
For Sale    4918
Name: count, dtype: int64